In [17]:
import numpy as np
import cascade as csc
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../")))

from cascade_breakup_integration import (
    ParticleState,
    CollisionFragmentHandler,
    CollisionAwareSimulation,
    add_fragments_to_simulation
)

from nasa_breakup_wrapper import generate_fragments
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Set paths
BREAKUP_MODEL_PATH = "/home/andrea/LSMS_project/NASA-breakup-model-cpp/build_iridium_cosmos/breakupModel"
MIN_CHARACTERISTIC_LENGTH = 0.05  # 5 cm minimum fragment size


In [18]:
# Example: Create a two-satellite collision scenario at 800 km altitude
# Semi-major axis for 800 km altitude orbit
a_800km = 6.78e6  # meters (Earth radius 6.378e6 + 800 km)

# Create two satellites on slightly different orbits that will collide
# Satellite 1: circular orbit
obj1_id = 1000
obj1_mass = 500.0  # kg
obj1_semi_major_axis = a_800km
obj1_ecc = 0.0
obj1_incl = np.radians(45)  # 45 degrees
obj1_raan = 0.0
obj1_aop = 0.0
obj1_anomaly = 0.0

# Satellite 2: slightly elliptical orbit that crosses obj1's path
obj2_id = 1001
obj2_mass = 400.0  # kg
obj2_semi_major_axis = a_800km + 1000  # 1 km higher
obj2_ecc = 0.001
obj2_incl = np.radians(45.1)  # Slightly different inclination
obj2_raan = 0.0
obj2_aop = 0.0
obj2_anomaly = np.pi  # Start at opposite side of orbit

# Satellite 3
obj3_id = 1002
obj3_mass = 300.0  # kg
obj3_semi_major_axis = a_800km + 2000  # 2
obj3_ecc = 0.002
obj3_incl = np.radians(45.2)  # Slightly different inclination
obj3_raan = 0.0
obj3_aop = 0.0
obj3_anomaly = np.pi / 2  # Start at quarter orbit

# Convert orbital elements to Cartesian coordinates
# (You would use pykep or similar library for this)
# Example position and velocity vectors (in SI units)
obj1_pos = np.array([5000000000, 0.0, 0.0])  # meters
obj1_vel = np.array([3000, 0.0, 0.0])   # m/s (~7.7 km/s orbital velocity)

obj2_pos = np.array([5000000200, 0.0, 0.0])  # 100 km away
obj2_vel = np.array([-3000, 0.0, 0.0])  # Slightly faster velocity

obj3_pos = np.array([5000000900, 0.0, 0.0])  # 200 km away
obj3_vel = np.array([-1000, 0.0, 0.0])

# Define initial particle population
n_particles = 3
r_ic = np.array([
    obj1_pos,
    obj2_pos,
    obj3_pos
])

v_ic = np.array([
    obj1_vel,
    obj2_vel,
    obj3_vel
])

# Collision detection parameters
collision_radii = np.array([
    2.0,  # obj1 collision radius (2 m)
    1.5,  # obj2 collision radius (1.5 m)
    1.0   # obj3 collision radius (1 m)
])

# B* ballistic coefficient (for atmospheric models, 0 for outer space)
bstars = np.array([0.0, 0.0, 0.0])

# Create particle database
particle_db = {
    0: ParticleState(
        id=obj1_id,
        position=obj1_pos.copy(),
        velocity=obj1_vel.copy(),
        mass=obj1_mass,
        collision_radius=collision_radii[0],
        bstar=0.0,
        char_length=4.0,  # Characteristic length ~4m
        area_to_mass=0.04
    ),
    1: ParticleState(
        id=obj2_id,
        position=obj2_pos.copy(),
        velocity=obj2_vel.copy(),
        mass=obj2_mass,
        collision_radius=collision_radii[1],
        bstar=0.0,
        char_length=3.0,  # Characteristic length ~3m
        area_to_mass=0.035
    ),
    2: ParticleState(
        id=obj3_id,
        position=obj3_pos.copy(),
        velocity=obj3_vel.copy(),
        mass=obj3_mass,
        collision_radius=collision_radii[2],
        bstar=0.0,
        char_length=5.0,  # Characteristic length ~5m
        area_to_mass=0.033
    )
}

print(f"Initial configuration: {n_particles} particles")
print(f"  Object 1: mass={obj1_mass} kg, pos={obj1_pos}, vel={obj1_vel}")
print(f"  Object 2: mass={obj2_mass} kg, pos={obj2_pos}, vel={obj2_vel}")
print(f"  Object 3: mass={obj3_mass} kg, pos={obj3_pos}, vel={obj3_vel}")


Initial configuration: 3 particles
  Object 1: mass=500.0 kg, pos=[5.e+09 0.e+00 0.e+00], vel=[3000.    0.    0.]
  Object 2: mass=400.0 kg, pos=[5.0000002e+09 0.0000000e+00 0.0000000e+00], vel=[-3000.     0.     0.]
  Object 3: mass=300.0 kg, pos=[5.0000009e+09 0.0000000e+00 0.0000000e+00], vel=[-1000.     0.     0.]


In [19]:
# Define dynamics (Keplerian for simplicity, can add perturbations)
# Using CASCADE's default Earth mass
mu_earth = 3.986004418e14  # m^3/s^2

# For pure Keplerian dynamics
dyn = csc.dynamics.kepler()  # or use keplerian

# 1. Ensure state is 2D (N, 7)
state_matrix = np.column_stack((
    r_ic,            # N x 3
    v_ic,            # N x 3
    collision_radii  # N x 1
))

# 2. Ensure pars is 2D (N, n_pars)
# If bstars is [b1, b2, ...], this makes it [[b1], [b2], ...]
pars_matrix = np.array(bstars).reshape(-1, 1)

# 3. Define your collisional timestep
ct_value = 1.0

empty_pars = np.zeros((state_matrix.shape[0], 0))

# 4. Initialize the simulation
sim = csc.sim(
    state_matrix,
    ct_value,
    dyn=dyn,
    pars=empty_pars
)

print(f"CASCADE simulation initialized at t={sim.time}")
print(f"  Integrator tolerance: {sim.tol}")

CASCADE simulation initialized at t=0.0
  Integrator tolerance: 2.220446049250313e-16


In [20]:
import time
import numpy as np
from tqdm.auto import tqdm

# --- 1. Initialization ---
final_time = 10000
total_collisions_count = 0
next_frag_id = 50000
start_wall_clock = time.time()

for logger_name in logging.root.manager.loggerDict:
    logging.getLogger(logger_name).setLevel(logging.WARNING)

# Initialize the handler
collision_handler = CollisionFragmentHandler(
    min_char_length=MIN_CHARACTERISTIC_LENGTH,
    breakup_model_path=BREAKUP_MODEL_PATH
)
collision_handler.set_next_fragment_id(next_frag_id)

print(f"Starting Simulation: {len(particle_db)} objects | Target: {final_time}s")

# Use tqdm for a condensed, single-line progress bar
pbar = tqdm(total=final_time, unit="s", desc="Simulating")

while sim.time < final_time:
    
    outcome = sim.step()
    
    # Update progress bar position and description
    pbar.n = min(sim.time, final_time)
    pbar.set_postfix({
        "Objs": len(particle_db), 
        "Cols": total_collisions_count
    })
    pbar.refresh()

    # 1. Sync Python positions
    for i in range(len(sim.state)):
        if i in particle_db:
            particle_db[i].position = sim.state[i, 0:3]
            particle_db[i].velocity = sim.state[i, 3:6]

    if outcome == csc.outcome.collision:
        pi, pj = sim.interrupt_info
        total_collisions_count += 1
        pbar.write(f"\n[t={sim.time:.4f}s] COLLISION: {particle_db[pi].id} + {particle_db[pj].id}")
        
        # 2. Handle collision
        num_before_fragments = len(particle_db)
        new_particle_db, particles_to_remove = collision_handler.handle_collision(
            sim, pi, pj, particle_db
        )

        # 3. Identify survivors and fragments
        survivor_indices = [i for i in range(num_before_fragments) if i not in particles_to_remove]
        fragment_indices = [i for i in range(num_before_fragments, len(new_particle_db))]
        
        # 4. SYMPLECTIC DRIFT (Separation)
        # Using a semi-implicit Euler method for better energy conservation
        for f_idx in fragment_indices:
            frag = new_particle_db[f_idx]
            
            # 1. Accel from Earth's gravity
            r_vec = frag.position
            r_mag = np.sqrt(r_vec[0]**2 + r_vec[1]**2 + r_vec[2]**2)
            accel = - (mu_earth / r_mag**3) * r_vec
            
            # 2. Update Velocity
            frag.velocity += accel * 0.5
            
            # 3. Update Position (using updated velocity)
            frag.position += frag.velocity * 0.5

        # 5. Rebuild the simulation state
        r_list, v_list, rad_list = [], [], []
        updated_db = {}
        
        # Survivors (Current state from C++)
        for i, old_idx in enumerate(survivor_indices):
            p = new_particle_db[old_idx]
            r_list.append(sim.state[old_idx, 0:3])
            v_list.append(sim.state[old_idx, 3:6])
            rad_list.append(p.collision_radius)
            updated_db[i] = p
            
        # Drifting Fragments
        curr_idx = len(updated_db)
        for f_idx in fragment_indices:
            f_obj = new_particle_db[f_idx]
            r_list.append(f_obj.position)
            v_list.append(f_obj.velocity)
            rad_list.append(f_obj.collision_radius)
            updated_db[curr_idx] = f_obj
            curr_idx += 1
        
        # 6. Push back to C++
        if len(r_list) > 0:
            # Check for NaN - If r_mag was 0, gravity becomes infinite
            final_state = np.column_stack([np.array(r_list), np.array(v_list), np.array(rad_list)])
            
            if np.isnan(final_state).any():
                pbar.write("FATAL: NaN in state! Gravity calculation failed.")
                break

            empty_bst_list = [[] for _ in range(len(r_list))]
            
           
            sim.set_new_state_pars(final_state.tolist(), empty_bst_list)
            
            particle_db = updated_db
            pbar.write(f"Injected {len(fragment_indices)} fragments. Total: {len(particle_db)}")
        else:
            pbar.write("No particles left. Terminating.")
            break

pbar.close()

# -----------------------------
# Final summary
# -----------------------------
end_wall_clock = time.time()
print("\n" + "="*35)
print("SIMULATION FINISHED")
print(f"Total Collisions: {total_collisions_count}")
print(f"Wall Clock Runtime: {end_wall_clock - start_wall_clock:.2f} seconds")
print(f"Final Object Count: {len(particle_db)}")
print(f"Final Sim Time: {sim.time:.4f}s")
print("="*35)

Starting Simulation: 3 objects | Target: 10000s


Simulating:   0%|          | 0/10000 [00:00<?, ?s/s]


[t=0.0327s] COLLISION: 1000 + 1001
Running NASA Breakup Model simulation with config: /home/andrea/LSMS_project/NASA-breakup-model-cpp/collision_config.yaml
Simulation completed. Output CSV: /home/andrea/LSMS_project/NASA-breakup-model-cpp/fragments.csv
DEBUG: Separation distance: 300.3977m
DEBUG: Avg Vel: 3034.95 m/s | Max: 10225.63 m/s
Injected 2757 fragments. Total: 2758


KeyboardInterrupt: 

In [ ]:
import csv

with open("final_particle_state.csv", "w", newline="") as f:
    writer = csv.writer(f)

    writer.writerow([
        "index","id","x","y","z","vx","vy","vz",
        "mass","collision_radius","char_length","area_to_mass","bstar"
    ])

    for idx, p in particle_db.items():
        writer.writerow([
            idx,
            p.id,
            *p.position,
            *p.velocity,
            p.mass,
            p.collision_radius,
            p.char_length,
            p.area_to_mass,
            p.bstar
        ])

print("Saved final_particle_state.csv")